<a href="https://colab.research.google.com/github/ldaniel-hm/eml_tabular/blob/main/MonteCarloTodasLasVisitas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MC con Políticas epsilon-soft (off-policy)**

Este notebook describe un experimento de aprendizaje por refuerzo utilizando el algoritmo de Monte Carlo con políticas epsilon-soft. El propósito de este análisis es entrenar un agente en un entorno de gym con el juego "CliffWalking", un entorno estándar en el que el agente debe aprender a moverse a través de un mapa en busca de una meta, evitando obstáculos.

## **1. Preparación del Entorno**

La preparación consta de las siguientes partes:
- **Instalación de Dependencias**: Se instalan las librerías necesarias para utilizar el entorno `gymnasium` para la simulación, con el objetivo de crear un ambiente controlado para que el agente pueda interactuar.
- **Importación de Librerías**: Se importan las bibliotecas necesarias como `numpy` para el manejo de matrices y `matplotlib` para la visualización de los resultados.

- **Importación del Entorno "CliffWalking"**:
Se carga el entorno "CliffWalking".

- **Funciones para Mostrar los Resultados**: Se define una función para graficar la proporción de recompensas obtenidas en cada episodio del entrenamiento. Esto ayuda a visualizar el progreso del agente en términos de su desempeño durante el entrenamiento.



##### **Código de la Instalación e Importación**
----

In [ ]:
%%capture
#@title Instalamos gym
%pip install 'gym[box2d]==0.20.0'

## Instalación de algunos paquetes.
#!apt-get update
## Para usar gymnasium[box2d]
#!apt install swig
#!pip install gymnasium[box2d]

In [ ]:
from environments.cliff import make_cliff_env
from algorithms.mc_off_policy import MCOffPolicy
from utils.plotting import plot_curve

In [ ]:
NUM_EPISODES = 5000
SEED = 100

env = make_cliff_env(seed=SEED)

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)
# Definimos las acciones
UP, RIGHT, DOWN, LEFT = 0,1,2,3

## **2. Diseño del Agente**

El diseño del agente consta de dos partes: el algoritmo con el que aprende y las políticas (toma de decisiones) que realiza.

- **Políticas del Agente**
  - **Política epsilon-soft**: Se define una política donde todas las acciones tienen una probabilidad de ser elegida. Esta política se utiliza como behavior policy en el escenario off-policy.
  - **Política epsilon-greedy**: Basada en la política epsilon-soft. Permite al agente explorar con cierta probabilidad y explotar la mejor acción conocida con mayor probabilidad.
  - **Política greedy**: Es la target policy que el agente aprende y sigue para maximizar la recompensa esperada una vez completado el entrenamiento.

- **Algoritmo de Iteración de Valor**
  - Se implementa el algoritmo de Monte Carlo off-policy utilizando importancia de muestreo.
  - Se usa una $\epsilon$ constante para la política behavior, asegurando exploración suficiente y estabilidad en los pesos de importancia.
  - La actualización de Q se realiza de manera incremental considerando todos los retornos del episodio y ponderando por los pesos de importancia.

## **3. Experimentación**


- En esta sección, el algoritmo de Monte Carlo off-policy se ejecuta para el entorno de Cliff Walking.

- Se realiza un entrenamiento con un número determinado de episodios (5000 en concreto).

- La política que genera los episodios (behavior policy) es $\epsilon$-soft con $\epsilon$ constante, mientras que la target policy es greedy. No se aplica decaimiento de $\epsilon$ en este escenario para asegurar suficiente exploración y estabilidad de los pesos de importancia.

- Durante el entrenamiento hay una visualización de la recompensa total obtenida a lo largo de los episodios.

- Junto a dicho volcado se muestra gráficamente la recompensa por episodio y su tendencia (media móvil).

- También se hace un volcado de los valores Q de cada estado, donde se muestra cómo el agente valora diferentes acciones en distintos estados del entorno, lo que refleja su conocimiento sobre las mejores estrategias para alcanzar la meta sin caer en los agujeros.

- Además, se muestra la política óptima derivada de los valores Q, que es la que el agente seguiría si siempre eligiera la acción que maximiza su recompensa esperada.

### **3.1 Representaciones Gráficas**

Se mostrará la función $f(t) = \frac{\sum_{i=1}^t R_i}{t}$ para $t = 1,2,\ldots, NumeroEpisodios$. Aquí, $R_i$ es la recompensa obtenida en el episodio $i$. Como las recompensas son negativas por cada paso y caídas en el acantilado, esta función reflejará la calidad del aprendizaje, es decir, cuán eficientemente el agente alcanza el objetivo evitando el acantilado.

### **3.2 Experimentación**

- Se realizan 5000 episodios y se actualizan los valores Q (valor de acción) basándose en las recompensas obtenidas durante cada episodio completo (es decir, aplicamos Monte Carlo). Se aplica una política $\epsilon$-greedy sobre una política $\epsilon$-soft con un valor $\epsilon$ constante.

In [ ]:
# @title Aprendizaje
mc_off = MCOffPolicy(
    env,
    discount_factor=0.95,
    epsilon=0.3,
    max_steps=50
)

mc_off_rewards = mc_off.train(NUM_EPISODES)

In [ ]:
#@title Aciertos por número de episodios
plot_curve(
    mc_off_rewards,
    title="MC Off-Policy - Recompensa por episodio",
    ylabel="Recompensa total",
    moving_avg_window=200
)
print(f"Máxima proporcion: {mc_off_rewards[-1]}")

In [ ]:
# @title Tabla de valores Q
print("Valores Q para cada estado:\n", mc_off.Q)

In [ ]:
# @title Política final
pi_off_grid, actions_off, grid_off = mc_off.greedy_policy_analysis(mc_off.Q)
print("Política óptima obtenida\n", pi_off_grid)
print("Acciones", actions_off)
print("Para el siguiente grid\n", grid_off)

Política óptima obtenida
 [[0 0 2 0]
 [0 0 2 0]
 [0 0 0 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 0 0 3]
 [0 0 2 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 0 2 0]
 [0 0 2 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 0 0 0]
 [0 0 0 3]
 [0 0 0 3]
 [0 1 0 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 0 2 0]
 [0 1 0 0]
 [0 0 0 0]
 [0 0 0 3]
 [0 0 0 0]
 [0 0 0 3]
 [0 1 0 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 1 0 0]
 [0 0 2 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
Acciones 2, 2, 0, 1, 1, 3, 2, 1, 1, 1, 1, 2, 2, 1, 1, 0, 3, 3, 1, 1, 1, 1, 1, 2, 1, 0, 3, 0, 3, 1, 1, 1, 1, 1, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0
Para el siguiente grid
 o o o o o o o o o o o o
o o o o o o o o o o o o
o o o o o o o o o o o o
o C C C C C C C C C C x


## **4. Fin**